In [ ]:
import torch
from colpali_engine.models import ColModernVBert, ColModernVBertProcessor
from qdrant_client import QdrantClient
from qdrant_client.http import models
from pathlib import Path

# Configuration
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "colpali_collection" # Kembali ke collection versi original/regular
MODEL_NAME = "ModernVBERT/colmodernvbert"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")

# ColModernVBert Regular Search (No Quantization)
Notebook ini digunakan untuk melakukan pencarian tanpa optimasi kuantisasi sebagai pembanding.

In [ ]:
# 1. Load Model
model = ColModernVBert.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map=DEVICE,
    trust_remote_code=True
).eval()
processor = ColModernVBertProcessor.from_pretrained(MODEL_NAME)

# 2. Connect to Qdrant
client = QdrantClient(url=QDRANT_URL)
print(f"Connected to Qdrant. Collection: {COLLECTION_NAME}")

In [ ]:
# 3. Regular Search Function (Tanpa Quantization)
def search_regular(query_text, top_k=5):
    with torch.no_grad():
        query = processor.process_queries([query_text]).to(model.device)
        query_embedding = model(**query)
        query_vec = query_embedding[0].cpu().float().numpy().tolist()
        num_query_tokens = len(query_vec)
    
    # Pencarian standar pada collection regular
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        using="initial",
        limit=top_k
    ).points
    
    for res in results:
        res.normalized_score = res.score / num_query_tokens
    
    return results

def print_results(results):
    for i, res in enumerate(results):
        norm_score = getattr(res, 'normalized_score', 0)
        print(f"Rank {i+1} | Raw Score: {res.score:.2f} | Norm Score: {norm_score:.4f}")
        print(f"  PDF: {res.payload.get('pdf_name', 'N/A')} | Page: {res.payload.get('page', 'N/A')}")
        print("-" * 50)

In [ ]:
# 4. Test Regular Search
query = "Apple black rot"
print(f"Searching for: '{query}' (Regular Mode)\n")

import time
start = time.time()
results = search_regular(query)
end = time.time()

print(f"Search Time: {end - start:.4f} seconds\n")
print_results(results)